In [5]:
import pandas as pd

# Load the processed test dataset
test_df = pd.read_csv('test_with_gas.csv')

# Load the all_2024 dataset
all_2024_df = pd.read_csv('all_2024_cleaned.csv')

# Rename the columns in all_2024 for clarity
all_2024_df = all_2024_df.rename(columns={
    "Date (GMT+1)": "datetime",
    "Cross border electricity trading": "cross_border_trading",
    "Non-Renewable": "non_renewable",
    "Renewable": "renewable",
    "Load": "load",
    "CO2 Emission Allowances, Auction DE": "co2_emission_allowances"
})

# Convert the datetime column in all_2024 to timezone-aware datetime
all_2024_df['datetime'] = pd.to_datetime(all_2024_df['datetime'], format='%Y-%m-%d %H:%M:%S').dt.tz_localize('Europe/Berlin', ambiguous='infer')

# Fill missing values in all_2024
numeric_columns = [
    'cross_border_trading', 'non_renewable', 'renewable', 'load', 'co2_emission_allowances'
]
for column in numeric_columns:
    all_2024_df[column] = (
        all_2024_df[column].interpolate(method='linear')  # Linear interpolation
        .ffill()  # Forward fill
        .bfill()  # Backward fill
    )

# Create a temporary 'datetime' column in test_df for merging
test_df['datetime'] = pd.to_datetime(test_df['ds']).dt.tz_localize('Europe/Berlin', ambiguous='infer')

# Merge all_2024 data into test_df using the 'datetime' column
test_df = pd.merge(test_df, all_2024_df, on='datetime', how='left')

# Drop the temporary 'datetime' column from test_df
# Keep 'ds' as-is to preserve the original nature
test_df = test_df.drop(columns=['datetime'])

# Save the updated test dataset
test_df.to_csv('test_with_all.csv', index=False)

print("Hourly data from all_2024 successfully merged into test_with_gas without altering 'ds'.")

Hourly data from all_2024 successfully merged into test_with_gas without altering 'ds'.
